# Latvian Legal Changes Multi-Agent System

Interactive testing notebook for the multi-agent legal monitoring system built with Google's Agent Development Kit (ADK).

**What this notebook does:**
- Demonstrates ADK best practices: Sequential, Parallel, and Loop agents
- Tests individual specialist agents (Likumi, TAP, Saeima) 
- Runs complete end-to-end coordinator with quality validation
- Shows advanced features: progressive batching, session memory, workflow enforcement

**Two ways to test:**
1. **This notebook** - Python API with full control and debugging
2. **ADK Web UI** - Visual interface at `http://localhost:8000` (run `adk web` in terminal)

**Architecture:** See [README.md](README.md) for full system documentation and architecture details.

## Prerequisites

- Python 3.11+
- A `.env` file at project root with `GOOGLE_API_KEY=your_key` (get from [Google AI Studio](https://aistudio.google.com/app/api-keys))

**Choose your testing method:**
- **Notebook**: Run cells below in order (do not use "Run All" to avoid rate limits)
- **Web UI**: Skip to "Alternative: ADK Web UI" section, then run `adk web` in terminal

## ⚠️ Optional: Clean Cache

**WARNING:** This deletes all logs, cache, and previous run data. Only run if starting completely fresh and want to restart.

In [ ]:
import os
import shutil
import glob

# Clear cache directories and files
cache_paths = ["cache", "logs", "__pycache__", "venv", ".pytest_cache"]

for path in cache_paths:
    if os.path.isdir(path):
        shutil.rmtree(path)
        print(f"✅ Removed directory: {path}")

# Remove .pyc files and __pycache__ recursively
for pyc_file in glob.glob("**/*.pyc", recursive=True):
    os.remove(pyc_file)
for pycache_dir in glob.glob("**/__pycache__", recursive=True):
    if os.path.isdir(pycache_dir):
        shutil.rmtree(pycache_dir)

print("🧹 Cache cleanup complete")

In [ ]:
%pip install -q google-adk python-dotenv requests beautifulsoup4

## Setup: Configure API Key

Create a `.env` file in the project root with your Gemini API key:
```
GOOGLE_API_KEY=your_key_here
```

In [ ]:
import os
from dotenv import load_dotenv

# Load .env from project root
load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
if not GOOGLE_API_KEY:
    raise RuntimeError("Please set GOOGLE_API_KEY in a .env file at the project root or as an env var.")
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "FALSE"
print("✅ Loaded GOOGLE_API_KEY from .env")

In [ ]:
# Import ADK components
from google.adk.agents import Agent, SequentialAgent, ParallelAgent, LoopAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import google_search
from google.adk.plugins.logging_plugin import LoggingPlugin
from google.genai import types

# Configure retry options (ADK best practice)
retry_config = types.HttpRetryOptions(
    attempts=5,
    exp_base=7,
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504]
)

print("✅ ADK components imported successfully")

---
## Setup: Configure Logging

Set up DEBUG-level logging for observability (ADK best practice).

In [ ]:
# Import and configure logging (force reload to pick up fixes)
import importlib
import sys
if 'agents.utils.logging_config' in sys.modules:
    importlib.reload(sys.modules['agents.utils.logging_config'])

from agents.utils.logging_config import setup_logging, get_logger

# Set up logging at DEBUG level
logger = setup_logging(log_level="DEBUG", log_file="latvian_legal_debug.log")
logger.info("🚀 Starting Latvian Legal Changes Multi-Agent System")

---
## Alternative: ADK Web UI

The ADK includes a web UI for interactive testing and debugging. This is useful for:
- Visual inspection of agent execution flow
- Real-time observability of tool calls and state
- Testing without writing Python code

**To launch the web UI:**

Run in your terminal (not in notebook):
```bash
cd likumumekletajs
adk web adk_agents
```

Then open the URL shown.

**Using the Web UI:**

1. **Select Agent**: Choose from registered agents (coordinator, likumi, tap, saeima)
2. **Enter Query**: Type your query in Latvian (e.g., "Dokumenti par izglītību. Meklē laikposmā no 2025-04-10 līdz 2025-04-15")
3. **Monitor Execution**: Watch agent workflow, tool calls, and state updates in real-time
4. **View Logs**: See DEBUG-level logs in the UI and in `logs/latvian_legal_debug.log`

**Benefits of Web UI:**
- No code needed for testing
- Visual workflow inspection
- Session persistence across runs
- Real-time state monitoring
- Useful for demos and non-technical stakeholders

### Preparing Agents for Web UI

The ADK web UI discovers agents from your project's agent registry. Agents are already configured for this.

In [ ]:
# Verify agents are registered and ready for web UI
from agents.agent_registry import list_registered, get_agent, get_coordinator

print("🌐 Agents available in Web UI:\n")
registered = list_registered()
print(f"   Specialists: {', '.join(registered)}")
print(f"   Coordinator: available via get_coordinator()")

print("\n💡 To test in Web UI:")
print("   1. Run in terminal: cd likumumekletajs && adk web adk_agents")
print("   2. Open http://localhost:8000 in browser")
print("   3. Select agent from dropdown (coordinator, likumi, tap, saeima)")
print("   4. Enter query in Latvian with date range")
print("   5. Watch execution flow in real-time")

---
## Specialist Agent Testing: Likumi

Test the Likumi.lv specialist agent in isolation. See [README.md](README.md) for full architecture details.

**Query Format:** `{topic}. Meklē laikposmā no {start_date} līdz {end_date}`

**Agent Features:**
- Custom tools: `likumi_scraper`, `fetch_likumi_document`
- Session memory: `get_cached_summary`, `store_document_summary`
- Workflow enforcement: `check_workflow_complete()` ensures 100% coverage
- Progressive batching: Processes 8-10 documents per turn to avoid context overflow

In [ ]:
# Optional: Force reload after code changes
import sys
import importlib

modules_to_reload = [
    'agents.likumi_agent',
    'agents.agent_registry',
    'agents.tools.enforcement_wrapper',
    'agents.tools.workflow_check',
    'agents.tools.likumi_scraper',
    'agents.tools.memory_tools',
]

for module in modules_to_reload:
    if module in sys.modules:
        del sys.modules[module]

from agents.agent_registry import reset_agent
reset_agent('likumi')
print("✅ Cleared LikumiAgent cache")

In [ ]:
# Initialize Likumi specialist via registry pattern
from agents.agent_registry import get_agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.plugins.logging_plugin import LoggingPlugin

likumi_agent = get_agent('likumi')
session_service = InMemorySessionService()

test_runner = Runner(
    agent=likumi_agent,
    session_service=session_service,
    plugins=[LoggingPlugin()],
    app_name="latvian_legal_monitor"
)

print("✅ LikumiAgent initialized")
print(f"   Tools: likumi_scraper, fetch_likumi_document, memory, workflow")
print(f"   Session: InMemorySessionService (shared state)")
print(f"   Logs: logs/latvian_legal_debug.log")

In [ ]:
# Test Likumi agent with small date range
response = await test_runner.run_debug(
    "Dokumenti par bērnu tiesībām. Meklē laika posmā no 2025-07-10 līdz 2025-07-15"
)

print("\n" + "=" * 70)
print("LIKUMI AGENT RESPONSE:")
print("=" * 70)
print(response)

---
## Specialist Agent Testing: TAP Portal

Test the TAP Portal specialist agent in isolation.

**Features:**
- 6 document types: Legal acts, meetings, public participation, tasks, declassified docs, notices
- Multi-date field support: Submission, deadlines, publication dates
- Empty keyword search + post-fetch concept expansion for high recall

In [ ]:
# Optional: Force reload TAP agent
import sys

tap_modules = [
    'agents.tap_agent',
    'agents.tools.tap_scraper',
    'agents.tools.fetch_tap_document',
]

for module in tap_modules:
    if module in sys.modules:
        del sys.modules[module]

from agents.agent_registry import reset_agent
reset_agent('tap')
print("✅ Cleared TAPAgent cache")

In [ ]:
# Initialize TAP agent
from agents.agent_registry import get_agent

tap_agent = get_agent('tap')
tap_session_service = InMemorySessionService()

tap_runner = Runner(
    agent=tap_agent,
    session_service=tap_session_service,
    plugins=[LoggingPlugin()],
    app_name="latvian_legal_monitor"
)

print("✅ TAPAgent initialized")
print(f"   Document types: 6 (acts, meetings, participation, tasks, declassified, notices)")
print(f"   Strategy: Empty keyword + concept expansion")

In [ ]:
# Test TAP agent
response = await tap_runner.run_debug(
    "Kultūras nozares aktualitātes. Meklē laika posmā no 2025-10-22 līdz 2025-10-23"
)

print("\n" + "=" * 70)
print("TAP AGENT RESPONSE:")
print("=" * 70)
print(response)

---
## Specialist Agent Testing: Saeima

Test the Saeima.lv parliamentary specialist agent.

**Features:**
- 5 document types: Legislation, decisions, questions, requests, committee meetings
- Chronological estimation: Linear interpolation + 100% buffer for edge cases
- 83% parliamentary coverage (agendas excluded)

In [ ]:
# Force reload Saeima agent modules
import sys
import importlib

saeima_modules = [
    'agents.saeima_agent',
    'agents.tools.saeima_scraper',
    'agents.tools.fetch_saeima_document',
]

for module in saeima_modules:
    if module in sys.modules:
        del sys.modules[module]
        print(f"🔄 Removed {module} from cache")

# Reset Saeima agent in registry
from agents.agent_registry import reset_agent
reset_agent('saeima')
print("✅ Cleared SaeimaAgent cache - next get_agent('saeima') will rebuild with latest code")


In [ ]:
# Initialize Saeima Agent via Registry
from agents.agent_registry import get_agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.plugins.logging_plugin import LoggingPlugin

# Get Saeima agent from registry
saeima_agent = get_agent('saeima')

# Create separate session service for Saeima testing
saeima_session_service = InMemorySessionService()

# Create runner for Saeima agent
saeima_runner = Runner(
    agent=saeima_agent,
    session_service=saeima_session_service,
    plugins=[LoggingPlugin()],
    app_name="latvian_legal_monitor"
)

print("✅ Saeima Agent initialized via registry")
print(f"   Agent: SaeimaAgent (specialist)")
print(f"   Session: InMemorySessionService (shared state with Likumi & TAP)")
print(f"   Tools: saeima_scraper, fetch_saeima_document, memory tools")
print(f"   Document Types: 5 (legislation, decisions, questions, requests, committees)")
print(f"   Implementation: Simple OpenView pattern (330 lines)")
print(f"   Enforcement: Workflow validation enabled")
print(f"\n📝 Ready to test Saeima.lv queries")

In [ ]:
# Test Saeima Agent with date range
response = await saeima_runner.run_debug(
    "Izmaiņas autovadītājiem. Meklē laikposmā no 2025-04-20 līdz 2025-04-26"
)

print("\n" + "=" * 70)
print("📊 SAEIMA AGENT RESPONSE:")
print("=" * 70)
print(response)

---
## Full System Testing: Complete Coordinator

Test the end-to-end multi-agent system. See [README.md](README.md) for full architecture documentation.

**Architecture (ADK Patterns):**
1. **ParallelAgent** - 3 specialists search simultaneously (Likumi, TAP, Saeima)
2. **SequentialAgent** - Aggregator normalizes results
3. **SequentialAgent** - Report pipeline with quality validation
   - TranslatorAgent (EN → LV)
   - **LoopAgent**
     - CriticAgent validates translation
     - RefinerAgent fixes issues or exits via `exit_report_loop()`

**Expected:** Professional Latvian report with source citations from all 3 sources.

In [ ]:
# Force reload all agents for fresh start
import sys
import importlib

all_modules = [
    'agents.likumi_agent',
    'agents.tap_agent',
    'agents.saeima_agent',
    'agents.aggregator_agent',
    'agents.report_agent',
    'agents.coordinator_agent',
    'agents.agent_registry',
    'agents.base_config',
]

for module in all_modules:
    if module in sys.modules:
        del sys.modules[module]
        print(f"🔄 Removed {module} from cache")

from agents.agent_registry import reset_all
reset_all()
print("✅ Cleared all agent caches - coordinator will rebuild with latest code")

In [ ]:
# Initialize Full Coordinator
from agents.agent_registry import get_coordinator
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.plugins.logging_plugin import LoggingPlugin

# Get full coordinator from registry
coordinator = get_coordinator()

# Create session service
coordinator_session = InMemorySessionService()

# Create runner
coordinator_runner = Runner(
    agent=coordinator,
    session_service=coordinator_session,
    plugins=[LoggingPlugin()],
    app_name="latvian_legal_monitor"
)

print("✅ Full coordinator initialized")
print(f"\n🏗️ Architecture:")
print(f"   SequentialAgent [")
print(f"     ParallelAgent [")
print(f"       LikumiAgent (specialist)")
print(f"       TAPAgent (specialist)")
print(f"       SaeimaAgent (specialist)")
print(f"     ]")
print(f"     AggregatorAgent (normalizer + first draft)")
print(f"     SequentialAgent [")
print(f"       TranslatorAgent (EN → LV)")
print(f"       LoopAgent(max=3) [")
print(f"         CriticAgent (validates)")
print(f"         RefinerAgent (fixes or exits)")
print(f"       ]")
print(f"     ]")
print(f"   ]")
print(f"\n📝 Ready for end-to-end test")

In [ ]:
# Run end-to-end test (small date range)
response = await coordinator_runner.run_debug(
    "Aktualitātes zvejniecībā. Meklē laikposmā no 2025-04-10 līdz 2025-04-14"
)

print("\n" + "=" * 70)
print("FINAL LATVIAN REPORT:")
print("=" * 70)
print(response)
print("\n" + "=" * 70)
print("✅ End-to-end test complete!")
print("=" * 70)

---
## Advanced Features

### Progressive Batching

The system's critical innovation: handles 50+ documents by processing in batches of 8-15 per turn.

**Why:** Large result sets (50+ documents) cause context overflow → `MALFORMED_FUNCTION_CALL` errors

**Solution:** Agent instructions enforce batch size limits. Validated with 69-document query → 4 batches → 100% success.

**Try:** Query with longer date range (7+ days) to see batching in action. Check logs for enforcer tracking.

In [ ]:
# Example: Query that triggers batching (7-day range)
response = await coordinator_runner.run_debug(
    "Vides aizsardzība. Meklē laikposmā no 2025-04-01 līdz 2025-04-07"
)
# Expected behavior:
# - 50-100+ documents discovered across 3 sources
# - Agent processes in 3-5 batches (check logs)
# - WorkflowEnforcer validates 100% coverage
# - Final report includes all relevant documents

print("📊 Monitor logs/latvian_legal_debug.log for enforcer tracking:")
print(response)

### Session Memory Inspection

View cached documents from the current session (demonstrates per-source memory architecture).

In [ ]:
# Inspect session state to see cached documents
session_id = coordinator_runner.create_session().id
session = coordinator_session.get_session(session_id)

if session and session.state:
    print("📦 Cached Documents by Source:\n")
    
    for key in ['temp:likumi_docs', 'temp:tap_docs', 'temp:saeima_docs']:
        if key in session.state:
            docs = session.state[key]
            source = key.split(':')[1].replace('_docs', '').upper()
            print(f"{source}: {len(docs)} documents cached")
            if docs:
                print(f"  Sample: {list(docs.keys())[0][:80]}...")
            print()
else:
    print("No session data yet - run a query first")

### Workflow Enforcement

The `WorkflowEnforcer` singleton ensures agents process ALL discovered documents before responding.

**Pattern:**
1. Scraper tools register result count: `enforcer.register_search_results(count)`
2. Fetch/cache tools increment counter: `enforcer.register_fetch_call()`
3. Agent must call `check_workflow_complete()` before final response
4. Returns `allowed=False` if documents remain unprocessed

**Benefit:** Prevents incomplete analysis (validated: 69/69 documents in production test).

In [ ]:
# Check enforcer state
from agents.tools.enforcement_wrapper import get_enforcer

enforcer = get_enforcer()
allowed, reason = enforcer.can_complete()

print("🔍 Workflow Enforcer Status:\n")
print(f"  Search results registered: {enforcer.search_results_count}")
print(f"  Fetch/cache calls made: {enforcer.fetch_calls_count}")
print(f"  Can complete: {allowed}")
print(f"  Reason: {reason}")

# Check logs for detailed tracking
print("\n💡 See logs/latvian_legal_debug.log for full enforcer tracking:")
print("   - 🔍 Enforcer: likumi_scraper returned X documents")
print("   - 📄 Enforcer: Fetch call #X/Y")
print("   - ✅ Enforcer: Workflow complete (Y documents processed)")